# Environment Smoke Test
Tests SpatialMETA, Scanpy, and Squidpy installation. All plots are saved to `./smoke_test_plots/` instead of displayed, for WSL compatibility.

In [ ]:
# Cell
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

In [ ]:
# Cell 1 - Version check
import scanpy as sc
import squidpy as sq
import spatialmeta as smt

sc.logging.print_header()
print(f'squidpy=={sq.__version__}')
print(f'spatialmeta=={smt.__version__}')
print('\n✓ All three packages imported successfully')

In [ ]:
# Cell 2 - Scanpy smoke test
print('--- Scanpy smoke test ---')

# Downloads ~50MB on first run, cached after that
adata = sc.datasets.visium_sge(sample_id='V1_Human_Lymph_Node')
adata.var_names_make_unique()
print(f'  Loaded: {adata.shape[0]} spots x {adata.shape[1]} genes')

# Preprocessing pipeline
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata)
print('  Preprocessing pipeline complete')

# Plot 1: PCA variance ratio
sc.pl.pca_variance_ratio(adata, n_pcs=15)
#save_plot('scanpy_pca_variance_ratio.png')

# Plot 2: UMAP with leiden clusters
sc.pl.umap(adata, color='leiden')
#save_plot('scanpy_umap_leiden.png')

# Plot 3: Spatial plot
sc.pl.spatial(adata, color='leiden')
#save_plot('scanpy_spatial_leiden.png')

print(f'\n✓ Scanpy OK — {adata.shape[0]} spots processed, 3 plots saved')

In [ ]:
# Cell 3 - Squidpy smoke test
print('--- Squidpy smoke test ---')

adata_sq = sq.datasets.visium_hne_adata()
print(f'  Loaded: {adata_sq.shape[0]} spots x {adata_sq.shape[1]} genes')

# Spatial graph and neighbourhood enrichment
sq.gr.spatial_neighbors(adata_sq, coord_type='grid')
sq.gr.nhood_enrichment(adata_sq, cluster_key='cluster')
print('  Spatial graph and neighbourhood enrichment complete')

# Plot 1: Neighbourhood enrichment
sq.pl.nhood_enrichment(adata_sq, cluster_key='cluster', figsize=(5, 4))

# Plot 2: Spatial scatter
sq.pl.spatial_scatter(adata_sq, color='cluster')

print(f'\n✓ Squidpy OK — spatial graph built, 2 plots saved')

In [ ]:
help(smt.data.load_adata)


In [ ]:
print(smt.data.list_datasets())


In [ ]:
# Cell 4 - SpatialMETA smoke test (paper dataset)
print('--- SpatialMETA smoke test ---')

# List available datasets from the paper
print('  Available datasets:')
print(smt.data.list_datasets())

# Load the ST (spatial transcriptomics) AnnData
# Adjust dataset key to match list_datasets() output above
adata_joint = smt.data.load_adata(sample_name='Y7_T_raw', modality='joint')
print(f'  Loaded ST+SM data: {adata_joint.shape}')

# Plot: first gene in spatial context
sc.pl.spatial(adata_joint, color=adata_joint.var_names[0])

print(f'\n✓ SpatialMETA OK — paper dataset loaded, 1 plot saved')

In [ ]:
# Cell 4b - SpatialMETA minimal pipeline test
print('--- SpatialMETA pipeline test ---')

import spatialmeta as smt
import scanpy as sc

# Use already loaded joint object
adata = adata_joint.copy()  # work on a copy to preserve the original
print(f'  Input: {adata.shape}')
print(f'  Obs columns: {list(adata.obs.columns[:5])}')  # peek at metadata
print(f'  Obsm keys: {list(adata.obsm.keys())}')        # check spatial coords exist

# Step 1: Normalise
smt.pp.normalize_total_joint_adata_sm_st(adata)
print('  ✓ Normalisation complete')

# Step 2: Find spatially variable features
smt.pp.spatial_variable_joint_adata_sm_st(adata)
print('  ✓ Spatially variable features identified')

# Step 3: Integration model
model = smt.model.ConditionalVAESTSM(adata)
model.fit(max_epoch=10)  # 10 epochs just to confirm it runs, not for real results
print('  ✓ Model trained')

# Step 4: Get latent embedding
adata.obsm['X_spatialmeta'] = model.get_latent_embedding()
print(f'  ✓ Latent representation shape: {adata.obsm["X_spatialmeta"].shape}')

# Step 5: Cluster on the latent space
sc.pp.neighbors(adata, use_rep='X_spatialmeta')
sc.tl.leiden(adata, resolution=0.5)
print(f'  ✓ Clustering complete — {adata.obs["leiden"].nunique()} clusters found')

# Plot: spatial map of clusters
sc.pl.spatial(adata, color='leiden')

print('\n✓ SpatialMETA pipeline OK')

In [ ]:
# Cell 5 - Cross-package integration check
print('--- Cross-package integration check ---')

adata_combined = sq.datasets.visium_hne_adata_crop()

# Pass a Squidpy-loaded object through a full Scanpy pipeline
sc.pp.normalize_total(adata_combined)
sc.pp.log1p(adata_combined)
sc.pp.pca(adata_combined, n_comps=10)
sc.pp.neighbors(adata_combined)
sc.tl.umap(adata_combined)
print('  Scanpy pipeline on Squidpy data complete')

# Plot: UMAP coloured by cluster
sc.pl.umap(adata_combined, color='cluster', show=False)

print(f'\n✓ Scanpy <-> Squidpy interop OK, 1 plot saved')